# Data types

## Motivating example

Let's load the [Dr. Who dataset]() from [TidyTuesday](https://github.com/rfordatascience/tidytuesday).

In [ ]:
library('dplyr')
library('ggplot2')

In [ ]:
dr_who = read.csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/refs/heads/main/data/2023/2023-11-28/drwho_episodes.csv')

dr_who |> glimpse()

The `first_aired` variable tells us when each episode aired. **How can we find the most viewed episodes by year?** 

In [ ]:
dr_who |>
    mutate(first_aired = as.Date(first_aired), year = lubridate::year(first_aired)) |>
    group_by(year) |>
    slice_max(uk_viewers, n = 1) |>
    select(year, episode_title, first_aired, uk_viewers) |> 
    display()

---

## Examples from *Hello Data Science*

The code below is adapted from Chapter 6, "Strings, Dates, and Factors", of:

> Doğucu, M., Medina, C., & Castro, A. *Hello Data Science: A Friendly
> Introduction with Applications.* <https://hellodata.science/chapters/chapter-06>

Licensed CC BY-NC-ND 4.0. Only change: `library(tidyverse)` becomes the
individual packages this browser kernel provides.

### The data

In [ ]:
library('sfemergency25')
library('stringr')
library('lubridate')
library('forcats')

data(sf911)
glimpse(sf911)

### Strings

In [ ]:
sf911 |> 
  mutate(
    address_long = str_replace(
      address, 
      pattern = "ST", 
      replacement = "STREET"
    )
  ) |>
  select(address_long) |> 
  head(30)

In [ ]:
sf911 |> 
  mutate(
    address_long = str_replace_all(
      address, 
      pattern = "ST", 
      replacement = "STREET"
    )
  ) |>
  select(address_long)

In [ ]:
address_key <- 
  c(
    "ST"   = "STREET", 
    "AVE"  = "AVENUE", 
    "PL"   = "PLACE", 
    "TER"  = "TERRACE"
  )

sf911 |> 
  mutate(address_long = str_replace_all(address, pattern = address_key)) |>
  select(address_long)

In [ ]:
address_key <- c(
  "\\bST\\b"  = "STREET",
  "\\bAVE\\b" = "AVENUE",
  "\\bPL\\b"  = "PLACE",
  "\\bTER\\b" = "TERRACE"
)

sf911 |> 
  mutate(address_long = str_replace_all(address, pattern = address_key)) |> 
  select(address_long)

In [ ]:
sf911 |> 
  mutate(address = str_to_title(address)) |>
  select(address) |> 
  head()

In [ ]:
sf911 |> 
  mutate(
    description = str_glue(
      "Incident {call_number} occurred in {neighborhoods_boundaries}"
    )
  ) |> 
  select(description) |> 
  head(3)

In [ ]:
sf911 |>
  mutate(is_fire_related = str_detect(call_type, pattern = "Fire|Smoke")) |>
  select(call_type, is_fire_related) |> 
  slice(1:10)

In [ ]:
str_sub("Hello Data Science", start = 1, end = 7)

### Dates

In [ ]:
sf911 |> 
  select(received_dt_tm) |> 
  head(3)

In [ ]:
sf911 <-
  sf911 |>
  mutate(received_dt_tm = ymd_hms(received_dt_tm))

str(sf911$received_dt_tm)

In [ ]:
sf911 <-
  sf911 |> 
  mutate(
    received_dt_tm = with_tz(received_dt_tm, tzone = "America/Los_Angeles")
  )

sf911 |> 
  select(received_dt_tm) |> 
  head(3)

In [ ]:
sf911 |> 
  mutate(
    hour_val = hour(received_dt_tm),
    month_val = month(received_dt_tm, label = TRUE),
    day_name = wday(received_dt_tm, label = TRUE)
  ) |> 
  select(received_dt_tm, hour_val, month_val, day_name) |> 
  head(3)

In [ ]:
sf911 |>
  mutate(
    goal_arrival_time = received_dt_tm + dseconds(480)
  ) |>
  select(received_dt_tm, goal_arrival_time) |>
  head(3)

In [ ]:
event_start <- ymd_hms("2026-03-07 05:00:00", tz = "America/Los_Angeles")

event_start + ddays(1)

event_start + days(1)

In [ ]:
sf911 <- 
  sf911 |> 
  mutate(
    on_scene_dt_tm = ymd_hms(on_scene_dt_tm),
    response_time = on_scene_dt_tm - received_dt_tm
  )

str(sf911$response_time)

In [ ]:
today()

now()

### Factors

In [ ]:
sf911 <-
  sf911 |>
  mutate(unit_type = as.factor(unit_type)) 

str(sf911$unit_type)

In [ ]:
levels(sf911$unit_type)

In [ ]:
student_data <- data.frame(
  year = c(
    "Senior", "First-Year", "Junior", "Sophomore", "Junior", "First-Year"
  )
) |>
  mutate(year = as.factor(year))

In [ ]:
ggplot(sf911, aes(y = unit_type)) +
  geom_bar()

In [ ]:
sf911 <- 
  sf911 |> 
  mutate(unit_type = fct_infreq(unit_type))

str(sf911$unit_type)

In [ ]:
sf911 |>
  mutate(unit_type = fct_rev(unit_type)) |>
  ggplot(aes(y = unit_type)) +
  geom_bar()

In [ ]:
sf911 |>
  mutate(unit_type = fct_lump_n(unit_type, n = 3)) |>
  ggplot(aes(y = unit_type)) +
  geom_bar()

In [ ]:
sf911 |>
  mutate(unit_type = fct_relevel(unit_type, "ENGINE")) |>
  group_by(unit_type) |>
  summarize(total_als_unit = sum(als_unit))

In [ ]:
sf911 |>
  mutate(unit_type = fct_relevel(unit_type, "ENGINE", "TRUCK")) |>
  group_by(unit_type) |>
  summarize(total_als_unit = sum(als_unit))

In [ ]:
sf911 <- sf911 |>
  mutate(
    neighborhoods_boundaries = as.factor(neighborhoods_boundaries),
    neighborhoods = fct_reorder(neighborhoods_boundaries, response_time)
  ) 

#sf911 |> pull(neighborhoods_boundaries)
sf911 |> pull(neighborhoods) |> levels()